<a href="https://colab.research.google.com/github/MonikaBarget/DigitalHistory/blob/master/ScrapingYouTubeComments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting YouTube comments via API

This script collects YouTube comments for all videos in a YouTube playlist, using the Google API. The code is based on the instructions provided in the following Github repository:

https://github.com/rodolflying/youtube_automation/blob/master/main.py

This version of the code is meant for execution in **Google Colab** and asks users to ingest both their Google API and their playlist URL via console input. The collected data are written to the user's Google Drive but also downloaded as a .zip archive.

In [ ]:
# Import required packages

from google.colab import drive, files
from googleapiclient.discovery import build
import pandas as pd
from time import sleep
import traceback
import os
import re
import zipfile
from datetime import datetime

print("Installations done!")

In [ ]:
# Connect your Google Drive

print("Connecting Google Drive...")
drive.mount('/content/drive')

print("Google Drive connected.")

In [ ]:
# Get user input

api_key = input("Please paste your YouTube API key here: ").strip()
playlist_url = input("Please paste your YouTube playlist URL here: ").strip()

In [ ]:
# Collecting and saving YouTube comments

## Define output directory on Google Drive

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

base_output_folder = f"/content/YouTube_comments_{timestamp}"
os.makedirs(base_output_folder, exist_ok=True)

drive_output_folder = f"/content/drive/MyDrive/YouTube_comments_{timestamp}"
os.makedirs(drive_output_folder, exist_ok=True)

zip_filename = f"YouTube_comments_{timestamp}.zip"

## Functions for data collection

def get_playlist_id(playlist_url):
    playlist_id_match = re.search(r"list=([^&]+)", playlist_url)

    if playlist_id_match:
        return playlist_id_match.group(1)
    else:
        raise ValueError("Invalid playlist URL")


def get_video_ids_from_playlist(api_key, playlist_id):
    youtube = build('youtube', 'v3', developerKey=api_key)

    video_ids = []

    request = youtube.playlistItems().list(
        part="snippet",
        playlistId=playlist_id,
        maxResults=100
    )

    while request:

        try:
            response = request.execute()

            for item in response['items']:
                video_id = item['snippet']['resourceId']['videoId']
                video_ids.append(video_id)

            request = youtube.playlistItems().list_next(request, response)

        except Exception as e:
            print(f"Error while fetching video IDs: {str(e)}")
            print(traceback.format_exc())
            break

    return video_ids


def get_comments(api_key, video_id):

    youtube = build('youtube', 'v3', developerKey=api_key)

    request = youtube.commentThreads().list(
        part="snippet,replies",
        videoId=video_id,
        textFormat="plainText",
        maxResults=100
    )

    df = pd.DataFrame(
        columns=['comment', 'replies', 'date', 'user_name']
    )

    while request:

        replies = []
        comments = []
        dates = []
        user_names = []

        try:

            response = request.execute()

            for item in response['items']:

                # comment text
                comment = item['snippet']['topLevelComment']['snippet']['textDisplay']
                comments.append(comment)

                # username
                user_name = item['snippet']['topLevelComment']['snippet']['authorDisplayName']
                user_names.append(user_name)

                # date
                date = item['snippet']['topLevelComment']['snippet']['publishedAt']
                dates.append(date)

                # replies
                replycount = item['snippet']['totalReplyCount']

                if replycount > 0:

                    replies.append([])

                    for reply in item['replies']['comments']:
                        reply_text = reply['snippet']['textDisplay']
                        replies[-1].append(reply_text)

                else:
                    replies.append([])

            # create dataframe
            df2 = pd.DataFrame({
                "comment": comments,
                "replies": replies,
                "user_name": user_names,
                "date": dates
            })

            df = pd.concat([df, df2], ignore_index=True)

            # save CSV
            output_file = f"{base_output_folder}/{video_id}_user_comments.csv"

            df.to_csv(
                output_file,
                index=False,
                encoding='utf-8'
            )

            print(f"Comments saved for video ID: {video_id}")

            sleep(2)

            request = youtube.commentThreads().list_next(
                request,
                response
            )

            if request:
                print("Please be patient. Collecting next batch of comments...")

        except Exception as e:

            print(f"Error occurred for video {video_id}: {str(e)}")
            print(traceback.format_exc())

            print("Waiting 10 seconds before continuing...")
            sleep(10)

            error_output = f"{base_output_folder}/{video_id}_partial_comments.csv"

            df.to_csv(
                error_output,
                index=False,
                encoding='utf-8'
            )

            break


# Main script

try:

    print("Reading playlist information...")

    playlist_id = get_playlist_id(playlist_url)

    print("Fetching video IDs from playlist...")

    video_ids = get_video_ids_from_playlist(
        api_key,
        playlist_id
    )

    print(f"Found {len(video_ids)} videos.")

    # Collect comments
    for video_id in video_ids:

        print("\n-----------------------------------")
        print(f"Processing video: {video_id}")

        get_comments(api_key, video_id)

    print("\nAll comments scraped successfully.")

    # Copy collected files to Google Drive

    print("\nCopying files to Google Drive...")

    for filename in os.listdir(base_output_folder):

        source = os.path.join(base_output_folder, filename)
        destination = os.path.join(drive_output_folder, filename)

        with open(source, 'rb') as src_file:
            with open(destination, 'wb') as dst_file:
                dst_file.write(src_file.read())

    print(f"Files saved to Google Drive folder:")
    print(drive_output_folder)

    # Create ZIP archive for download

    print("\nCreating ZIP file...")

    zip_path = f"/content/{zip_filename}"

    with zipfile.ZipFile(zip_path, 'w') as zipf:

        for filename in os.listdir(base_output_folder):

            file_path = os.path.join(base_output_folder, filename)

            zipf.write(
                file_path,
                arcname=filename
            )

    print("ZIP file created.")

    # Download ZIP archive

    print("\nStarting download...")

    files.download(zip_path)

    print("Download should begin automatically.")

except Exception as e:

    print("A critical error occurred.")
    print(str(e))
    print(traceback.format_exc())

This code was last updated in May 2026. I may require changes again in the future to still perform as expected. Please always consult the official YouTube API documentation when in doubt.

Monika Barget, Maastricht University